# Reddit Topic Modeling API Notebook

This notebook demonstrates the **internal API layer** for the project (functions in `reddit_utils.py`).
It shows: text cleaning, preparing training data, training an unsupervised fastText model, and producing a document embedding.


In [ ]:
!pip -q install fasttext nltk pandas numpy scikit-learn

In [ ]:
import nltk
nltk.download("stopwords")

import numpy as np

from reddit_utils import (
    get_english_stopwords,
    clean_text,
    write_training_file,
    train_fasttext_unsupervised,
    average_vector,
)


In [ ]:
stopwords_set = get_english_stopwords()

sample_comment = "Breaking: Government announces new climate policy! Visit https://news.com/article for details."
cleaned = clean_text(sample_comment, stopwords_set=stopwords_set)

print("Original:\n", sample_comment)
print("\nCleaned:\n", cleaned)


In [ ]:
toy_corpus = [
    "The government passed a new law on climate change.",
    "The stock market fell sharply after the election.",
    "Citizens protested for better education policies.",
    "A new technology startup raised millions in funding.",
    "Sports fans celebrated the championship victory!",
]

toy_clean = [clean_text(x, stopwords_set=stopwords_set) for x in toy_corpus]
train_path = write_training_file(toy_clean, out_path="api_demo_training.txt")

print("Wrote training data to:", train_path)
print("First lines:")
with open(train_path, "r", encoding="utf-8") as f:
    for _ in range(3):
        print("  ", f.readline().strip())


In [ ]:
ft_model = train_fasttext_unsupervised(
    train_txt_path=train_path,
    model_type="skipgram",  # or "cbow"
    dim=50,
    epoch=10,
    lr=0.05,
    min_count=1,
    thread=2,
    save_path="api_demo_fasttext.bin",
)

print("Model trained. Dimension:", ft_model.get_dimension())
print("Saved model to api_demo_fasttext.bin")


In [ ]:
vec = average_vector(ft_model, toy_clean[0])
print("Embedding shape:", vec.shape)
print("Preview:", vec[:10])


In [ ]:
word = "government"
print(f"Nearest neighbors for '{word}':")
for w, score in ft_model.get_nearest_neighbors(word, k=5):
    print(f"  {w:15s} {score:.3f}")
